# Codebase Q&A assistant: Colab runner
Thin notebook: all logic lives in `src/codeqa`, every step is a script (see README_details.md).
Use a **T4 GPU** runtime (Runtime > Change runtime type). Steps: install → test → prepare → benchmark → evaluate →
test queries → safety tests → (optional) model comparison → demo / peer testing → download results.

In [ ]:
# Option A: from GitHub (replace with your repo URL)
# !git clone https://github.com/<you>/<your-repo>.git /content/codebase-qa
# Option B: upload the zip and unzip it
# from google.colab import files; files.upload()
# !unzip -q codebase-qa.zip -d /content
%cd /content/codebase-qa

In [ ]:
!pip -q install -e '.[retrieval,llm,judge,ui,dev]'

In [ ]:
# Judge key (needed only for scripts/evaluate.py --judge / --pairwise). Store it in Colab Secrets.
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [ ]:
!python -m pytest -q        # 65 unit tests, no GPU / network needed

## 1. Prepare: clone Flask at the benchmark's commit, chunk, embed, build the graph
Should print **1409 chunks**. The first run downloads Qwen3-Embedding-0.6B (about 1.2 GB).

In [ ]:
!python scripts/prepare_repo.py --repo flask

## 1b. Justify the embedder: retrieval-only comparison
No generator, so it takes minutes. Compares Qwen3-Embedding-0.6B (default) with MiniLM and CodeRankEmbed on how often the retrieved context contains the files and identifiers named in the SWE-QA reference answers. Keep `reports/retrieval_comparison_flask.md` for the report.


In [ ]:
!python scripts/compare_retrieval.py --repo flask --configs configs/default.yaml configs/minilm.yaml configs/coderank.yaml


## 2. Benchmark: 48 SWE-QA questions, same generator for every strategy
`baseline` = plain top-6 RAG; `baseline_budget` = fair control (same context size); `code_seeds` = code-first
ordering without the graph; `structure` = the proposed method. Try `--limit 5` first; re-running resumes.

In [ ]:
!python scripts/run_experiment.py --repo flask --strategy baseline baseline_budget code_seeds structure --limit 5

In [ ]:
!python scripts/run_experiment.py --repo flask --strategy baseline baseline_budget code_seeds structure

## 3. Evaluate (paired comparisons are made against `baseline_budget`)

In [ ]:
!python scripts/evaluate.py --repo flask --strategies baseline baseline_budget code_seeds structure

In [ ]:
# LLM judge + pairwise (spends API calls). Judge = Gemini (different family from the Qwen generator).
!python scripts/evaluate.py --repo flask --strategies baseline_budget code_seeds structure --judge --pairwise

## 4. Test query set (24 hand-written Flask queries) and safety tests (12)
Each writes `reports/default/<name>_flask.{md,csv,jsonl}`. Download the CSVs, fill `manual_success` and `notes`.

In [ ]:
!python scripts/run_test_queries.py --repo flask --strategy baseline_budget structure

In [ ]:
!python scripts/run_probes.py --repo flask --strategy baseline_budget structure

## 5. Optional: ablations and a second generator
Edge-type ablations, then a smaller model under its own tag (results never mix across tags).

In [ ]:
# !python scripts/run_experiment.py --repo flask --strategy structure_refs structure_calls
# !python scripts/evaluate.py --repo flask --strategies baseline_budget structure structure_refs structure_calls
# !python scripts/run_experiment.py --repo flask --strategy baseline_budget structure --config configs/small_model.yaml --tag qwen1.5b
# !python scripts/evaluate.py --repo flask --strategies default/structure qwen1.5b/structure --base default/structure

## 6. Demo and peer testing
One question from the command line, or a web UI with a public link you can send to classmates.
Every tester's questions, answers and ratings go to `results/user_testing/<name>.jsonl`.

In [ ]:
!python scripts/ask.py --repo flask "What happens after Flask.wsgi_app is called?"

In [ ]:
# Stop this cell when testing is done.
!python scripts/app.py --repo flask --share

## 7. Download everything for the report

In [ ]:
!zip -qr results_and_reports.zip results reports
from google.colab import files; files.download("results_and_reports.zip")

## Peek at a result

In [ ]:
import json
rows = [json.loads(l) for l in open('results/default/flask__structure.jsonl')]
r = rows[0]
print(r['question']); print([(x['origin'], x['path'], x['start'], x['end']) for x in r['retrieved']]); print(r['answer'][:1000])